<div style="font-size: 1em; display: flex; align-items: center; gap: 8px; padding: 8px 16px; background: #F8F9FA; border-bottom: 2px solid #E0E0E0; margin: 0; line-height: 1">
    <img src="https://cdn.simpleicons.org/databricks/FF3621" width="24" height="24"/>
    <div style="color: #666">
        <span style="font-weight: bold; color: #333">Databricks Performance Optimization
</span>
        <span style="margin-left: 8px; color: #999">|</span>
        <span style="margin-left: 8px">04 Demo - Liquid Clustering vs Partitioning</span>
    </div>
</div>

![DB Academy](https://files.training.databricks.com/binder/prod_main/databricks-performance-optimization-en_us-3.0.1/images/20261106T135523Z/Databricks Performance Optimization/Includes/Images/common/databricks_academy_logo.png)

# Demo - Liquid Clustering vs Partitioning
#### Duration: 25 minutes

## Overview

Partitioning forces a physical layout, chosen at write time, on <b>one</b> or a <b>few low-cardinality columns</b>. 

Pick the wrong columns and every query that filters on something else reads the whole table.

**Liquid clustering** replaces that decision. You cluster by the columns you actually filter on (up to four), you can change those keys later without rewriting data, and you can even let `CLUSTER BY AUTO` pick and evolve them from your query history.

In this demo you will run the same filter against a **date-partitioned table** and a **liquid-clustered table**, then compare `system_vw.query.history` to see the partitioned table open all 730 files while the clustered table reads a fraction of the data. 

Then you will hand over key selection to the platform with `CLUSTER BY AUTO`.

## Learning Objectives

By the end of this demo, you'll be able to:

1. **Contrast liquid clustering with partitioning and ZORDER** and name the rare cases partitioning still survives.
2. **Cluster a table** with `CLUSTER BY`, choose good keys, and redefine them without rewriting data.
3. **Use `CLUSTER BY AUTO`** and state its prerequisites (UC managed plus Predictive Optimization).
4. **Explain `OPTIMIZE` vs `OPTIMIZE FULL`** and when a full recluster is required.
5. **Verify the layout win** in `system_vw.query.history` (`read_files`, `read_bytes`, `total_task_duration_ms`) and use `REPLACE WHERE` for selective overwrites.

<div style="border-left:4px solid #FFAB00;background:rgba(255,171,0,0.08);padding:14px 18px;border-radius:4px;margin:16px 0;font-family:sans-serif;">
  <strong style="display:block;color:#B8860B;margin-bottom:6px;font-size:15pt;">Pre-Run Timing Note</strong>
  <div style="color:#0b2026;font-size:14pt;line-height:1.55;">
    This notebook reads from <code>system_vw.query.history</code>, which lags live activity by <strong>5 to 10 minutes</strong>. Run the query cells first, then continue with the explanatory sections. By the time you reach the system-table cells, the rows will have landed. If a cell returns no rows, wait a few minutes and re-run it.
  </div>
</div>

## REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="
  border-left: 4px solid #f44336;
  background: #ffebee;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Select a Compute Environment</strong>
  <div style="color:#333;">

Before starting this notebook, select the required compute environment listed below.

- **Serverless Compute, Version 5**

![Serverless Select](https://files.training.databricks.com/binder/prod_main/databricks-performance-optimization-en_us-3.0.1/images/20261106T135523Z/Databricks Performance Optimization/Includes/Images/common/select-serverless.png)
<br></br>
  - How to select an environment version:
[AWS](https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/compute/serverless/dependencies#-select-an-environment-version) |
[GCP](https://docs.databricks.com/gcp/en/compute/serverless/dependencies#-select-an-environment-version)

**NOTE:**  This notebook requires **Python and was tested using Serverless V5**. The entire demo (both the "before" legacy path and the "after" managed table path) runs on serverless. Other compute may work but is not guaranteed to behave the same or support all features demonstrated.
  </div>
</div>

## A. Classroom Setup

Run the setup below. It builds a dedicated large base table (`lc_events_base`) and a date-partitioned copy (`events_partitioned`) for the "before". 

It exposes the table names and `DEMO_FILTER_CUSTOMER`, the non-partition column value the demo filters on.

In [ ]:
%run ./Includes/Classroom-Setup-04


## B. Three Ways to Lay Out a Table
<br/>
<div class="cw">
<style>
.cw { font-family: sans-serif; max-width: 1100px; margin: 0 auto; color: #0b2026; }
.cw * { box-sizing: border-box; }
.rc-hint { width: 100%; text-align: center; font-size: 15pt; color: #618794; font-style: italic; margin-bottom: 16px; }
.rc-hint strong { color: #FF5F46; font-style: normal; font-weight: 700; }
.rc-row { display: block; white-space: nowrap; font-size: 0; width: 100%; }
.rc-box { display: inline-block; width: 32%; margin-right: 2%; min-height: 118px; background: #F9F7F4; border-top: 8px solid transparent; border-left: 2px solid transparent; border-right: 2px solid transparent; border-bottom: 2px solid transparent; border-radius: 8px; padding: 18px 16px; text-align: center; cursor: pointer; user-select: none; vertical-align: top; transition: transform 0.12s; box-shadow: 0 2px 8px rgba(27,49,57,0.06); white-space: normal; }
.rc-box:last-child { margin-right: 0; }
.rc-box:hover { transform: translateY(-2px); }
.rc-box.active { background: #fff; border-left-color: var(--pc); border-right-color: var(--pc); border-bottom-color: var(--pc); }
.rc-label { display: block; font-weight: 700; font-size: 14pt; letter-spacing: 2px; text-transform: uppercase; line-height: 1.2; pointer-events: none; }
.rc-title { display: block; font-size: 18pt; font-weight: 700; color: #0b2026; line-height: 1.3; margin-top: 6px; pointer-events: none; }
.rc-sub { display: block; font-size: 14px; color: #5E7077; line-height: 1.35; margin-top: 6px; pointer-events: none; }
.rc-detail-wrap { overflow: hidden; max-height: 0; opacity: 0; transition: max-height 0.4s ease, opacity 0.3s ease, margin-top 0.3s ease; margin-top: 0; width: 100%; }
.rc-detail-wrap.open { max-height: 1600px; opacity: 1; margin-top: 16px; }
.rc-detail-card { background: #FFFFFF; border-radius: 10px; padding: 24px 28px; border-top: 7px solid #ccc; box-shadow: 0 2px 8px rgba(27,49,57,0.06); }
.rc-detail-title { font-size: 20pt; font-weight: 700; margin-bottom: 4px; line-height: 1.3; }
.rc-detail-sub { font-size: 14pt; color: #618794; margin-bottom: 18px; }
.rc-detail-body ul { margin: 0; padding-left: 22px; font-size: 15pt; line-height: 1.6; color: #0b2026; }
.rc-detail-body li { margin-bottom: 10px; }
.rc-detail-body code { font-family: monospace; font-size: 14pt; background: #EEEDE9; padding: 1px 6px; border-radius: 3px; color: #1B5162; }
/* file grid */
.fg { border-collapse: collapse; width: auto; max-width: 720px; margin: 0 auto; }
.fg th { background: #0b2026; color: #FFFFFF; font-size: 14px; font-weight: 700; padding: 10px 6px; text-align: center; border: 1px solid #FFFFFF; }
.fg th.rh { background: #1B5162; text-align: left; padding-left: 14px; }
.fg td { border: 1px solid #EEEDE9; height: 56px; text-align: center; vertical-align: middle; background: #FFFFFF; }
.fg td.rh { background: #1B5162; color: #FFFFFF; font-size: 14px; font-weight: 700; text-align: left; padding-left: 14px; white-space: nowrap; }
.fg .files { display: flex; flex-wrap: wrap; gap: 3px; justify-content: center; align-items: center; }
.sq { background: #FF5F46; border-radius: 2px; display: inline-block; }
.big { width: 30px; height: 30px; }
.mid { width: 19px; height: 19px; }
.sm { width: 11px; height: 11px; }
.mini { width: 8px; height: 8px; }
.tgt { width: 22px; height: 22px; }
.fg td.ingest { background: rgba(255,171,0,0.08); border: 3px solid #FFAB00; }
.fg td.freshcol { border: 3px solid #FFAB00; background: rgba(255,171,0,0.08); }
.clus { display: inline-flex; flex-wrap: wrap; gap: 3px; justify-content: center; align-items: center; border: 3px solid #00A972; background: rgba(0,169,114,0.06); border-radius: 6px; padding: 5px 7px; }
.hourbox { display: inline-flex; flex-wrap: wrap; gap: 3px; justify-content: center; align-items: center; border: 3px solid #FFAB00; background: rgba(255,171,0,0.08); border-radius: 6px; padding: 5px 7px; }
.leg { font-size: 14px; color: #618794; }
</style>

<div class="rc-hint">Click each layout to see how it stores the <strong>same data</strong>.</div>

<div class="rc-row">
  <span class="rc-box" data-id="0" onclick="rcSelect(0)" style="--pc:#618794; border-top-color:#618794;">
    <span class="rc-label" style="color:#618794;">Legacy</span>
    <span class="rc-title">Partitioning</span>
    <span class="rc-sub">One folder per value &rarr; small-file explosion</span>
  </span><span class="rc-box" data-id="1" onclick="rcSelect(1)" style="--pc:#618794; border-top-color:#618794;">
    <span class="rc-label" style="color:#618794;">Legacy</span>
    <span class="rc-title">ZORDER</span>
    <span class="rc-sub">Multi-column skipping, but rewrites every run</span>
  </span><span class="rc-box" data-id="2" onclick="rcSelect(2)" style="--pc:#00A972; border-top-color:#00A972;">
    <span class="rc-label" style="color:#00A972;">New default</span>
    <span class="rc-title">Liquid clustering</span>
    <span class="rc-sub">Same data kept at the right file size</span>
  </span>
</div>

<div class="rc-detail-wrap" id="rc-detail-wrap">
  <div class="rc-detail-card" id="rc-detail-card"></div>
</div>

</div>

<script>
var REVEAL = [
  { title: "Hive-style partitioning", tcolor: "#FFAB00", sub: "Low-cardinality columns, fixed at write time. Each column is one event_date folder; each row is a customer. Feb 8 is freshest: many small writes, no automatic compaction.",
    html:
      "<table class='fg'>"
      + "<tr><th style='background:#0b2026; text-align:left; padding-left:14px;'>PARTITION (event_date)</th><th>Feb 5</th><th>Feb 6</th><th>Feb 7</th><th style='background:#FFAB00; color:#0b2026;'>Feb 8, fresh ingest</th></tr>"
      + "<tr><td class='rh'>Customer A</td><td><div class='files'><span class='sq mid'></span></div></td><td><div class='files'><span class='sq mid'></span></div></td><td><div class='files'><span class='sq mid'></span></div></td><td class='freshcol'><div class='files'><span class='sq mini'></span><span class='sq mini'></span><span class='sq mini'></span><span class='sq mini'></span><span class='sq mini'></span></div></td></tr>"
      + "<tr><td class='rh'>Customer B</td><td><div class='files'><span class='sq sm'></span></div></td><td><div class='files'><span class='sq sm'></span></div></td><td><div class='files'><span class='sq sm'></span></div></td><td class='freshcol'><div class='files'><span class='sq mini'></span><span class='sq mini'></span><span class='sq mini'></span></div></td></tr>"
      + "<tr><td class='rh'>Customer C</td><td><div class='files'><span class='sq big'></span><span class='sq big'></span></div></td><td><div class='files'><span class='sq big'></span><span class='sq big'></span></div></td><td><div class='files'><span class='sq big'></span><span class='sq big'></span></div></td><td class='freshcol'><div class='files'><span class='sq mini'></span><span class='sq mini'></span><span class='sq mini'></span><span class='sq mini'></span><span class='sq mini'></span><span class='sq mini'></span><span style='font-size:11px; color:#618794; font-weight:700; margin-left:4px; white-space:nowrap;'>24 files</span></div></td></tr>"
      + "<tr><td class='rh'>Customer D</td><td><div class='files'><span class='sq sm'></span></div></td><td></td><td><div class='files'><span class='sq sm'></span></div></td><td class='freshcol'></td></tr>"
      + "<tr><td class='rh'>Customer E</td><td><div class='files'><span class='sq sm'></span></div></td><td><div class='files'><span class='sq sm'></span></div></td><td><div class='files'><span class='sq sm'></span></div></td><td class='freshcol'><div class='files'><span class='sq sm'></span></div></td></tr>"
      + "<tr><td class='rh'>Customer F</td><td><div class='files'><span class='sq mid'></span></div></td><td><div class='files'><span class='sq mid'></span></div></td><td><div class='files'><span class='sq mid'></span></div></td><td class='freshcol'><div class='files'><span class='sq mini'></span><span class='sq mini'></span><span class='sq mini'></span></div></td></tr>"
      + "</table>"
      + "<div style='display:flex; gap:22px; flex-wrap:wrap; margin-top:18px;'>"
      + "<div class='rc-detail-body' style='flex:1 1 360px;'><ul>"
      + "<li>Physical <strong>folders per partition value</strong>.</li>"
      + "<li>Prunes only when you filter the <strong>partition column</strong>.</li>"
      + "<li>Frequent small writes landing in the same partition folder cause a <strong>small-file explosion</strong>: Feb 8 shows 24 tiny files for one customer that day, versus just a few larger files on Feb 5 to Feb 7. Nothing consolidates them automatically, so they stay fragmented.</li>"
      + "<li>Layout is <strong>locked in</strong> without a rewrite.</li>"
      + "</ul></div>"
      + "<div style='flex:0 0 auto; align-self:center;' class='leg'><span class='sq mid' style='vertical-align:middle;'></span> = one file &#183; bigger square = bigger file</div>"
      + "</div>" },
  { title: "ZORDER", tcolor: "#4299E0", sub: "A clause on OPTIMIZE that sorts and colocates values from one or more columns into the same files, so filters on those columns skip more data. Each run rewrites the matching files, and effectiveness drops after a few columns.",
    html:
      "<div style='display:flex; gap:16px; align-items:center; justify-content:center; flex-wrap:wrap; margin-bottom:8px;'>"
      + "<div style='text-align:center;'>"
      + "<div style='display:grid; grid-template-columns:repeat(4,26px); gap:4px; justify-content:center;'>"
      + "<span style='width:26px;height:26px;background:#4299E0;border-radius:3px;'></span><span style='width:26px;height:26px;background:#FFAB00;border-radius:3px;'></span><span style='width:26px;height:26px;background:#4299E0;border-radius:3px;'></span><span style='width:26px;height:26px;background:#FF5F46;border-radius:3px;'></span>"
      + "<span style='width:26px;height:26px;background:#FFAB00;border-radius:3px;'></span><span style='width:26px;height:26px;background:#FF5F46;border-radius:3px;'></span><span style='width:26px;height:26px;background:#FFAB00;border-radius:3px;'></span><span style='width:26px;height:26px;background:#4299E0;border-radius:3px;'></span>"
      + "<span style='width:26px;height:26px;background:#FF5F46;border-radius:3px;'></span><span style='width:26px;height:26px;background:#4299E0;border-radius:3px;'></span><span style='width:26px;height:26px;background:#FF5F46;border-radius:3px;'></span><span style='width:26px;height:26px;background:#FFAB00;border-radius:3px;'></span>"
      + "</div>"
      + "<div class='leg' style='margin-top:8px;'>Before: values scattered</div>"
      + "</div>"
      + "<div style='display:flex; flex-direction:column; align-items:center; text-align:center; color:#98102A;'>"
      + "<div style='font-size:22pt; font-weight:700;'>OPTIMIZE ZORDER</div>"
      + "<div style='font-size:26pt; font-weight:800;'>&#8635;</div>"
      + "<div class='leg' style='color:#98102A;'>full rewrite every run</div>"
      + "</div>"
      + "<div style='text-align:center;'>"
      + "<div style='display:grid; grid-template-columns:repeat(4,26px); gap:4px; justify-content:center;'>"
      + "<span style='width:26px;height:26px;background:#4299E0;border-radius:3px;'></span><span style='width:26px;height:26px;background:#4299E0;border-radius:3px;'></span><span style='width:26px;height:26px;background:#4299E0;border-radius:3px;'></span><span style='width:26px;height:26px;background:#FFAB00;border-radius:3px;'></span>"
      + "<span style='width:26px;height:26px;background:#FFAB00;border-radius:3px;'></span><span style='width:26px;height:26px;background:#FFAB00;border-radius:3px;'></span><span style='width:26px;height:26px;background:#FF5F46;border-radius:3px;'></span><span style='width:26px;height:26px;background:#FF5F46;border-radius:3px;'></span>"
      + "<span style='width:26px;height:26px;background:#FF5F46;border-radius:3px;'></span><span style='width:26px;height:26px;background:#FF5F46;border-radius:3px;'></span><span style='width:26px;height:26px;background:#FF5F46;border-radius:3px;'></span><span style='width:26px;height:26px;background:#FF5F46;border-radius:3px;'></span>"
      + "</div>"
      + "<div class='leg' style='margin-top:8px;'>After: like values grouped</div>"
      + "</div>"
      + "</div>"
      + "<div class='rc-detail-body' style='margin-top:16px;'><ul>"
      + "<li><strong>Multi-column skipping</strong>, but <strong>rewrites on each run</strong>.</li>"
      + "<li><strong>Not incremental</strong>, must be re-applied as data changes.</li>"
      + "<li>Only for tables <strong>not yet on liquid clustering</strong>.</li>"
      + "</ul></div>" },
  { title: "Liquid clustering", tcolor: "#00A972", sub: "Same table, same rows. Files are automatically kept near the target size: heavy sections split finely, while fresh ingest lands mixed across customers until the next optimize pass splits it.",
    html:
      "<table class='fg'>"
      + "<tr><th class='rh'>Clustered by customer</th><th>Feb 5</th><th>Feb 6</th><th>Feb 7</th><th style='background:#FFAB00; color:#0b2026;'>Feb 8, fresh ingest</th></tr>"
      + "<tr><td class='rh'>Customer A</td><td></td><td rowspan='2'><div style='display:flex; align-items:center; justify-content:center; height:100%; transform: translate(-10px, -8px);'><span class='clus'><span style='display:flex; flex-direction:column; align-items:center;'><span class='sq tgt'></span><span style='font-size:8px; color:#0b2026;'>A</span></span><span style='display:flex; flex-direction:column; align-items:center;'><span class='sq tgt'></span><span style='font-size:8px; color:#0b2026;'>B</span></span><span style='font-size:10px; color:#00A972; font-weight:700;'>F2</span></span></div></td><td></td><td rowspan='6' class='ingest'><div style='display:flex; flex-direction:column; gap:8px; align-items:center; justify-content:center; height:100%;'><span class='sq tgt'></span><span class='sq tgt'></span><span style='font-size:13px; color:#B8860B; font-weight:700; text-align:center; line-height:1.35;'>fresh ingest,<br>not yet split</span></div></td></tr>"
      + "<tr><td class='rh'>Customer B</td><td></td><td></td></tr>"
      + "<tr><td class='rh'>Customer C</td><td><span class='clus'><span class='sq tgt'></span><span class='sq tgt'></span><span style='font-size:9px; color:#618794; font-weight:700; margin-left:2px; white-space:nowrap;'>2 files</span></span></td><td><span class='hourbox'><span class='sq tgt'></span><span class='sq tgt'></span><span class='sq tgt'></span><span class='sq tgt'></span><span style='font-size:9px; color:#0b2026; font-weight:700; margin-left:2px; white-space:nowrap;'>by hour</span></span></td><td><span class='clus'><span class='sq tgt'></span></span></td></tr>"
      + "<tr><td class='rh'>Customer D</td><td rowspan='2' style='padding:6px;'><div style='display:flex; align-items:center; justify-content:center; height:100%; transform: translateY(-8px);'><span class='clus'><span style='display:flex; flex-direction:column; align-items:center;'><span class='sq tgt'></span><span style='font-size:8px; color:#0b2026;'>D</span></span><span style='display:flex; flex-direction:column; align-items:center;'><span class='sq tgt'></span><span style='font-size:8px; color:#0b2026;'>E</span></span><span style='font-size:10px; color:#00A972; font-weight:700;'>F1</span></span></div></td><td></td><td></td></tr>"
      + "<tr><td class='rh'>Customer E</td><td></td><td><span class='clus'><span class='sq tgt'></span><span style='font-size:10px; color:#00A972; font-weight:700; margin-left:2px;'>F1</span></span></td></tr>"
      + "<tr><td class='rh'>Customer F</td><td></td><td></td><td></td></tr>"
      + "</table>"
      + "<div style='display:flex; gap:22px; flex-wrap:wrap; margin-top:18px;'>"
      + "<div class='rc-detail-body' style='flex:1 1 360px;'><ul>"
      + "<li>Up to <strong>four keys</strong>, <strong>high-cardinality friendly</strong>.</li>"
      + "<li><strong>Incremental</strong>, and keys change <strong>without a rewrite</strong>.</li>"
      + "<li><code>CLUSTER BY AUTO</code> can pick keys for you.</li>"
      + "<li><strong>Not compatible</strong> with partitioning or ZORDER.</li>"
      + "<li>New writes land mixed across all customers first, shown as the shared Feb 8 block. See <strong>B1</strong> below for a step-by-step walkthrough of how clustering splits and merges files over time.</li>"
      + "</ul></div>"
      + "<div style='flex:0 0 auto; align-self:center;' class='leg'><span style='display:inline-block; width:16px; height:16px; border:3px solid #00A972; border-radius:3px; vertical-align:middle;'></span> green box = one target-size file, already split and clustered by customer<br><span style='display:inline-block; width:16px; height:16px; border:3px solid #FFAB00; border-radius:3px; vertical-align:middle;'></span> amber box = one file split further by hour<br><span class='sq tgt' style='vertical-align:middle;'></span> = one file's worth of data. See <strong>B1</strong> for the full walkthrough.</div>"
      + "</div>" }
];
var rcCurrent = null;
function rcSelect(id) {
  var wrap = document.getElementById('rc-detail-wrap');
  var card = document.getElementById('rc-detail-card');
  var d = REVEAL[id];
  document.querySelectorAll('.rc-box').forEach(function(b) {
    b.classList.toggle('active', parseInt(b.dataset.id) === id && rcCurrent !== id);
  });
  if (rcCurrent === id) { wrap.classList.remove('open'); rcCurrent = null; return; }
  rcCurrent = id;
  card.style.borderTopColor = d.tcolor;
  card.innerHTML = '<div class="rc-detail-title" style="color:' + d.tcolor + ';">' + d.title + '</div><div class="rc-detail-sub">' + d.sub + '</div><div class="rc-detail-body">' + d.html + '</div>';
  wrap.classList.add('open');
}
</script>

##### EXPAND FOR ADDITIONAL DETAILS

<details>

### Why liquid clustering replaces the other two
- Partitioning commits you to one or a few low-cardinality columns, chosen when you create the table, and it only helps queries that filter those exact columns. Low cardinality means the column has few distinct values, such as date or region. Filter anything else and there is no pruning. A high-cardinality partition column also scatters data into tiny files, the exact problem from the last demo.
- ZORDER gives multi-column skipping, but it runs inside `OPTIMIZE` and rewrites data every time. It is not incremental, so it has to be re-applied, and it is now only for tables not yet migrated to liquid clustering.
- Liquid clustering supports up to four keys, handles high-cardinality columns, clusters incrementally (only new data is touched), and lets you change keys later without rewriting the table. The docs are explicit: clustering is not compatible with partitioning or ZORDER, so a table is one or the other.

### When partitioning still survives
- Only at very large scale. Databricks says do not partition under 1 TB, use liquid clustering from 1 TB to 100 TB, and consider partitioning only at 100 TB or more, and even then verify liquid clustering first.
- The other case is coarse isolation for management (for example a physical separation you drop or overwrite as a unit), and even that is shrinking as Delta gains selective-overwrite features.

</details>



## B1. Liquid Clustering in Action

Step through how the clustered table's file boundaries actually behave. Each step highlights one behavior, not a fixed rule about dates or customers.
<br /><br />
<div class="lca">
<style>
.lca { font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif; max-width: 900px; margin: 0 auto; color: #0b2026; }
.lca * { box-sizing: border-box; }
.lca-card { background: #FFFFFF; border-radius: 10px; padding: 28px 32px; border-top: 7px solid #00A972; box-shadow: 0 2px 8px rgba(27,49,57,0.06); text-align: center; min-height: 240px; }
.lca-stage { font-size: 13pt; font-weight: 700; letter-spacing: 1.5px; text-transform: uppercase; color: #00A972; margin-bottom: 8px; }
.lca-caption { font-size: 16pt; font-weight: 600; color: #0b2026; margin-bottom: 22px; line-height: 1.4; }
.lca-visual { display: flex; justify-content: center; align-items: center; gap: 10px; min-height: 90px; flex-wrap: wrap; }
.lca-sq { background: #FF5F46; border-radius: 3px; display: inline-block; }
.lca-hl { border: 3px solid #FFAB00; background: rgba(255,171,0,0.12); border-radius: 6px; padding: 8px 10px; display: inline-flex; flex-direction: column; gap: 6px; align-items: center; }
.lca-tag { font-size: 11pt; color: #0b2026; font-weight: 700; }
.lca-note { font-size: 12pt; color: #618794; margin-top: 18px; }
.lca-op { font-size: 18pt; color: #618794; font-weight: 700; }
.lca-result { font-size: 12pt; color: #00A972; font-weight: 700; }
.lca-nav { display: flex; justify-content: center; align-items: center; gap: 18px; margin-top: 20px; }
.lca-arrow { width: 40px; height: 40px; border-radius: 50%; background: #1B5162; color: #fff; font-size: 18pt; display: flex; align-items: center; justify-content: center; cursor: pointer; user-select: none; }
.lca-arrow:hover { background: #00A972; }
.lca-dots { display: flex; gap: 8px; }
.lca-dot { width: 10px; height: 10px; border-radius: 50%; background: #EEEDE9; cursor: pointer; }
.lca-dot.active { background: #00A972; }
</style>

<div class="lca-card" id="lca-card"></div>
<div class="lca-nav">
  <div class="lca-arrow" onclick="lcaStep(-1)">&#8249;</div>
  <div class="lca-dots" id="lca-dots"></div>
  <div class="lca-arrow" onclick="lcaStep(1)">&#8250;</div>
</div>
</div>

<script>
var LCA = [
  { stage: "Flexible file boundaries", caption: "A target-size file can span several dates for one small customer.",
    visual: "<div class='lca-hl'><span class='lca-sq' style='width:22px;height:22px;'></span><span class='lca-tag'>Feb 5</span></div><span class='lca-op'>+</span><div class='lca-hl'><span class='lca-sq' style='width:22px;height:22px;'></span><span class='lca-tag'>Feb 7</span></div><span class='lca-op'>&rarr;</span><span class='lca-result'>1 file</span>",
    note: "Customer E writes so little per day that two separate dates still fit in one target-size file." },
  { stage: "Flexible file boundaries", caption: "A target-size file can also span several customers on a single date.",
    visual: "<div class='lca-hl'><span class='lca-sq' style='width:22px;height:22px;'></span><span class='lca-tag'>Customer A</span></div><span class='lca-op'>+</span><div class='lca-hl'><span class='lca-sq' style='width:22px;height:22px;'></span><span class='lca-tag'>Customer B</span></div><span class='lca-op'>&rarr;</span><span class='lca-result'>1 file, Feb 6</span>",
    note: "Two typical customers on the same day share one file when neither alone is close to the target size." },
  { stage: "Flexible file boundaries", caption: "The same file can mix several dates and several tiny customers together.",
    visual: "<div class='lca-hl'><span class='lca-sq' style='width:18px;height:18px;'></span><span class='lca-tag'>D, Feb 5</span></div><span class='lca-op'>+</span><div class='lca-hl'><span class='lca-sq' style='width:18px;height:18px;'></span><span class='lca-tag'>E, Feb 5</span></div><span class='lca-op'>+</span><div class='lca-hl'><span class='lca-sq' style='width:18px;height:18px;'></span><span class='lca-tag'>E, Feb 7</span></div><span class='lca-op'>&rarr;</span><span class='lca-result'>1 file</span>",
    note: "Liquid clustering optimizes for target file size, not for a fixed date or customer boundary." },
  { stage: "Subdividing heavy partitions", caption: "A customer's busiest date gets subdivided by hour instead of staying one oversized file.",
    visual: "<span class='lca-sq' style='width:34px;height:34px;'></span><span class='lca-op'>&rarr;</span><div class='lca-hl'><span class='lca-sq' style='width:14px;height:14px;'></span><span class='lca-tag'>00 to 06h</span></div><div class='lca-hl'><span class='lca-sq' style='width:14px;height:14px;'></span><span class='lca-tag'>06 to 12h</span></div><div class='lca-hl'><span class='lca-sq' style='width:14px;height:14px;'></span><span class='lca-tag'>12 to 18h</span></div><div class='lca-hl'><span class='lca-sq' style='width:14px;height:14px;'></span><span class='lca-tag'>18 to 24h</span></div>",
    note: "Customer C's heaviest day is too large for one file, so clustering splits it further, by hour instead of by customer." },
  { stage: "Ingest first", caption: "At first, one file per ingest batch covers every customer at once.",
    visual: "<span class='lca-sq' style='width:40px;height:40px;'></span><span class='lca-tag' style='margin-left:12px;'>Feb 8, all customers</span>",
    note: "Fresh writes are not yet organized by customer. This is the shared ingest block from the table above." },
  { stage: "Split as data accumulates", caption: "As more data lands, the heaviest customer splits out into its own file first.",
    visual: "<span class='lca-sq' style='width:40px;height:40px;'></span><span class='lca-op'>&rarr;</span><div class='lca-hl'><span class='lca-sq' style='width:26px;height:26px;'></span><span class='lca-tag'>Customer C</span></div><div class='lca-hl'><span class='lca-sq' style='width:26px;height:26px;'></span><span class='lca-tag'>everyone else</span></div>",
    note: "Clustering prioritizes splitting where it saves the most scanning first, the heaviest customer." },
  { stage: "Maintenance cleans up", caption: "Later, table maintenance merges the last small leftover files into right-sized ones.",
    visual: "<span class='lca-sq' style='width:12px;height:12px;'></span><span class='lca-sq' style='width:12px;height:12px;'></span><span class='lca-sq' style='width:12px;height:12px;'></span><span class='lca-op'>&rarr;</span><div class='lca-hl'><span class='lca-sq' style='width:24px;height:24px;'></span><span class='lca-tag'>merged</span></div>",
    note: "This is the same OPTIMIZE pass that Predictive Optimization runs automatically on UC managed tables." }
];
var lcaCurrent = 0;
function lcaRender() {
  var d = LCA[lcaCurrent];
  document.getElementById('lca-card').innerHTML =
    '<div class="lca-stage">' + d.stage + ' (' + (lcaCurrent + 1) + ' of ' + LCA.length + ')</div>' +
    '<div class="lca-caption">' + d.caption + '</div>' +
    '<div class="lca-visual">' + d.visual + '</div>' +
    '<div class="lca-note">' + d.note + '</div>';
  var dotsHtml = '';
  for (var i = 0; i < LCA.length; i++) {
    dotsHtml += '<div class="lca-dot' + (i === lcaCurrent ? ' active' : '') + '" onclick="lcaGoto(' + i + ')"></div>';
  }
  document.getElementById('lca-dots').innerHTML = dotsHtml;
}
function lcaStep(delta) {
  lcaCurrent = (lcaCurrent + delta + LCA.length) % LCA.length;
  lcaRender();
}
function lcaGoto(i) { lcaCurrent = i; lcaRender(); }
lcaRender();
</script>

##### EXPAND FOR ADDITIONAL DETAILS

<details>

### Why file boundaries do not follow dates or customers
- Liquid clustering's only real constraint is the **target file size**. It packs rows together until a file reaches that size, whatever combination of dates or customers gets there.
- A small customer's rows from several different dates can end up in the same file, because none of those dates alone is worth a dedicated file.
- The same logic works across customers: several typical customers on one date can share a file if none of them alone reaches the target size.

### Why heavy partitions split by hour
- When one customer's single date is itself too large for one file, clustering has nothing coarser to split by within that date, so it subdivides by a finer-grained value, such as hour of day.
- This is the same skipping mechanism working at a different grain: instead of skipping whole dates or whole customers, the scan can skip specific hours.

### Why the three stages matter
- The **ingest first, split, cleanup** sequence shown here is the same incremental process covered in section D1. Fresh writes always land unorganized before `OPTIMIZE` has a chance to run.
- Predictive Optimization runs this maintenance automatically on UC managed tables, so these stages happen without a scheduled job.

</details>

## C. Before: The Partitioned Table Cannot Prune Our Filter

The table is partitioned by **event_date**, but the demo filters on **customer_idx**. The diagram shows why that combination reads everything.
<br /><br />
<div class="prunex">
  <style>
    .prunex { font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif; color: #1f2933; background: #ffffff; padding: 18px 16px 22px 16px; border: 1px solid #e2e8ee; border-radius: 10px; max-width: 1040px; margin: 0 auto; }
    .prunex h3 { margin: 0 0 4px 0; font-size: 19px; font-weight: 650; color: #14343f; letter-spacing: .1px; }
    .prunex p.sub { margin: 0 0 14px 0; font-size: 15px; color: #5f6b76; }
    .prunex svg { display: block; width: 100%; height: auto; }
    .prunex text { font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif; }
    .prunex .hdr { font-size: 14px; font-weight: 700; fill: #14343f; letter-spacing: .3px; }
    .prunex .sm { font-size: 12px; fill: #62707d; }
    .prunex .tag { font-size: 11px; font-weight: 700; letter-spacing: .4px; }
    .prunex .note { margin-top: 12px; font-size: 14px; color: #5f6b76; border-left: 3px solid #618794; padding: 4px 0 4px 10px; line-height: 1.55; }
    .prunex code { font-family: ui-monospace, SFMono-Regular, Menlo, Consolas, monospace; font-size: 14px; background: #f2f5f8; padding: 1px 4px; border-radius: 3px; color: #23424c; }
  </style>
  <h3>Why the Date Partitions Cannot Help</h3>
  <p class="sub">Partition pruning only fires when you filter the partition column. Here the layout key (<code>event_date</code>) and the filter key (<code>customer_idx</code>) are different, and this customer has activity on every day shown, so every partition folder holds a slice of their rows.</p>
  <svg viewBox="0 0 1040 420" role="img" aria-label="The table is split into date partition folders across the top. The filter is on customer, whose rows sit in every date folder, so every partition must be read.">
    <defs>
      <marker id="prunex-arr" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse">
        <path d="M 0 0 L 10 5 L 0 10 z" fill="#98102A"></path>
      </marker>
    </defs>
    <!-- layout row: partition folders by date -->
    <text class="hdr" x="16" y="28">PHYSICAL LAYOUT: one folder per event_date</text>
    <!-- 5 date-partition folders, each holding rows from many customers (colored bands) -->
    <g>
      <!-- folder template: outer rect + 4 customer bands, target customer band in coral -->
      <rect x="16" y="44" width="184" height="150" rx="8" fill="#f7fafc" stroke="#b9c7d2"></rect>
      <text class="tag" x="30" y="64" fill="#14343f">event_date = Feb 5</text>
      <rect x="30" y="74" width="156" height="20" rx="3" fill="#cfe8ec" stroke="#7fbcc8"></rect>
      <rect x="30" y="98" width="156" height="20" rx="3" fill="#f7dcd0" stroke="#e0a882"></rect>
      <rect x="30" y="122" width="156" height="20" rx="3" fill="#cfe8ec" stroke="#7fbcc8"></rect>
      <rect x="30" y="146" width="156" height="20" rx="3" fill="#d6ecdd" stroke="#8bbf9c"></rect>
      <rect x="30" y="170" width="156" height="16" rx="3" fill="#FF5F46" fill-opacity="0.88"></rect>
      <text class="tag" x="34" y="182" fill="#ffffff">customer 42 (1/5)</text>
    </g>
    <g>
      <rect x="212" y="44" width="184" height="150" rx="8" fill="#f7fafc" stroke="#b9c7d2"></rect>
      <text class="tag" x="226" y="64" fill="#14343f">event_date = Feb 6</text>
      <rect x="226" y="74" width="156" height="20" rx="3" fill="#d6ecdd" stroke="#8bbf9c"></rect>
      <rect x="226" y="98" width="156" height="20" rx="3" fill="#cfe8ec" stroke="#7fbcc8"></rect>
      <rect x="226" y="122" width="156" height="20" rx="3" fill="#f7dcd0" stroke="#e0a882"></rect>
      <rect x="226" y="146" width="156" height="20" rx="3" fill="#cfe8ec" stroke="#7fbcc8"></rect>
      <rect x="226" y="170" width="156" height="16" rx="3" fill="#FF5F46" fill-opacity="0.88"></rect>
      <text class="tag" x="230" y="182" fill="#ffffff">customer 42 (2/5)</text>
    </g>
    <g>
      <rect x="408" y="44" width="184" height="150" rx="8" fill="#f7fafc" stroke="#b9c7d2"></rect>
      <text class="tag" x="422" y="64" fill="#14343f">event_date = Feb 7</text>
      <rect x="422" y="74" width="156" height="20" rx="3" fill="#cfe8ec" stroke="#7fbcc8"></rect>
      <rect x="422" y="98" width="156" height="20" rx="3" fill="#d6ecdd" stroke="#8bbf9c"></rect>
      <rect x="422" y="122" width="156" height="20" rx="3" fill="#cfe8ec" stroke="#7fbcc8"></rect>
      <rect x="422" y="146" width="156" height="20" rx="3" fill="#f7dcd0" stroke="#e0a882"></rect>
      <rect x="422" y="170" width="156" height="16" rx="3" fill="#FF5F46" fill-opacity="0.88"></rect>
      <text class="tag" x="426" y="182" fill="#ffffff">customer 42 (3/5)</text>
    </g>
    <g>
      <rect x="604" y="44" width="184" height="150" rx="8" fill="#f7fafc" stroke="#b9c7d2"></rect>
      <text class="tag" x="618" y="64" fill="#14343f">event_date = Feb 8</text>
      <rect x="618" y="74" width="156" height="20" rx="3" fill="#f7dcd0" stroke="#e0a882"></rect>
      <rect x="618" y="98" width="156" height="20" rx="3" fill="#cfe8ec" stroke="#7fbcc8"></rect>
      <rect x="618" y="122" width="156" height="20" rx="3" fill="#d6ecdd" stroke="#8bbf9c"></rect>
      <rect x="618" y="146" width="156" height="20" rx="3" fill="#cfe8ec" stroke="#7fbcc8"></rect>
      <rect x="618" y="170" width="156" height="16" rx="3" fill="#FF5F46" fill-opacity="0.88"></rect>
      <text class="tag" x="622" y="182" fill="#ffffff">customer 42 (4/5)</text>
    </g>
    <g>
      <rect x="800" y="44" width="184" height="150" rx="8" fill="#f7fafc" stroke="#b9c7d2"></rect>
      <text class="tag" x="814" y="64" fill="#14343f">event_date = Feb 9</text>
      <rect x="814" y="74" width="156" height="20" rx="3" fill="#cfe8ec" stroke="#7fbcc8"></rect>
      <rect x="814" y="98" width="156" height="20" rx="3" fill="#f7dcd0" stroke="#e0a882"></rect>
      <rect x="814" y="122" width="156" height="20" rx="3" fill="#d6ecdd" stroke="#8bbf9c"></rect>
      <rect x="814" y="146" width="156" height="20" rx="3" fill="#cfe8ec" stroke="#7fbcc8"></rect>
      <rect x="814" y="170" width="156" height="16" rx="3" fill="#FF5F46" fill-opacity="0.88"></rect>
      <text class="tag" x="818" y="182" fill="#ffffff">customer 42 (5/5)</text>
    </g>
    <!-- filter band cutting across every folder -->
    <rect x="16" y="220" width="968" height="42" rx="8" fill="#fdf0f0" stroke="#e8b4b4"></rect>
    <text class="tag" x="32" y="246" fill="#98102A">FILTER: WHERE customer_idx = 42</text>
    <line x1="300" y1="241" x2="300" y2="200" stroke="#98102A" stroke-width="1.6" stroke-dasharray="4 3" marker-end="url(#prunex-arr)"></line>
    <line x1="496" y1="241" x2="496" y2="200" stroke="#98102A" stroke-width="1.6" stroke-dasharray="4 3" marker-end="url(#prunex-arr)"></line>
    <line x1="692" y1="241" x2="692" y2="200" stroke="#98102A" stroke-width="1.6" stroke-dasharray="4 3" marker-end="url(#prunex-arr)"></line>
    <text class="sm" style="fill:#98102A" x="540" y="246">customer 42 has activity every day, so no partition can be skipped</text>
    <!-- punchline -->
    <rect x="16" y="278" width="968" height="134" rx="8" fill="#14343f"></rect>
    <text x="32" y="308" style="font-size:20px;fill:#ffffff;font-family:-apple-system,BlinkMacSystemFont,Segoe UI,Roboto,Helvetica,Arial,sans-serif">Result: all partitions read, 0% pruned.</text>
    <text x="32" y="338" style="font-size:20px;fill:#ffffff;font-family:-apple-system,BlinkMacSystemFont,Segoe UI,Roboto,Helvetica,Arial,sans-serif">Partitioning only helps the query that filters the partition column.</text>
    <text x="32" y="368" style="font-size:20px;fill:#ffffff;font-family:-apple-system,BlinkMacSystemFont,Segoe UI,Roboto,Helvetica,Arial,sans-serif">Section D lays the same rows out by customer_idx instead,</text>
    <text x="32" y="398" style="font-size:20px;fill:#ffffff;font-family:-apple-system,BlinkMacSystemFont,Segoe UI,Roboto,Helvetica,Arial,sans-serif">so one customer lands in a tiny region of the table.</text>
  </svg>
  <div class="note">In the Query Profile, open the Scan operator and look at the percent of data pruned during scanning. Here it is 0 percent: the scan reads all 730 files to find a handful of matching rows. That is the classic partitioning trap, someone partitions by date because reports are usually by day, but the expensive queries filter by customer, product, or region.</div>
</div>
<br />
Run the query, then open the Query Profile to see zero data skipping.


<div style="border-left:4px solid #1B5162;background:#F8F9FC;padding:14px 18px;border-radius:4px;margin:16px 0;font-family:sans-serif;">
  <strong style="display:block;color:#1B5162;margin-bottom:6px;font-size:15pt;">Filter on a non-partition column</strong>
  <div style="color:#0b2026;font-size:14pt;">
    We run the same <code>WHERE customer_idx = ...</code> filter we will later run on the clustered table. 
    <br /><br />
    Because <code>customer_idx</code> is <strong>not</strong> the partition column, the date partitions cannot help. 
    <br /><br />
    After execution, open the Query Profile and note the Scan reads all files with zero pruning.
  </div>
</div>

In [ ]:
partitioned_q = f"""
SELECT event_type, COUNT(*) AS n
FROM {PARTITIONED_TABLE}
WHERE customer_idx = {DEMO_FILTER_CUSTOMER}
GROUP BY event_type
"""
print(partitioned_q)
spark.sql(partitioned_q).display()
print("Now open the Query Profile (See performance) and note the Scan reads all files with 0% pruned.")

##### EXPAND FOR ADDITIONAL DETAILS

<details>

### Reading the partitioned "before"
- The table is partitioned by `event_date`, but the filter is on `customer_idx`. Partition pruning only fires when you filter the partition column, so here it does nothing.
- In the Query Profile, open the Scan operator. You should see **0% pruned**: the scan reads all 730 files and skips none. Every file's per-column min/max range for `customer_idx` includes the filter value, so data skipping cannot eliminate a single file.
- The result: the engine scans the entire table (about 182.50 MB across 730 files) to find roughly a dozen matching rows. That is the worst-case read amplification.
- In section E we pull `read_files`, `read_bytes`, and `total_task_duration_ms` from `system_vw.query.history` for both queries. These columns give a stable comparison because they are not affected by compilation time or cluster warmup.
- This is the classic partitioning trap: someone partitioned by date because reports are "usually by day," but the expensive queries filter by customer, product, or region, and get no pruning at all.

</details>

## D. After: Cluster By What You Filter On

Create a UC managed table clustered by the columns the queries actually filter: `customer_idx`, `event_date`. 

No need for partitioning. Then we run the identical filter. 

Because the data is laid out by `customer_idx`, all rows for a given customer are co-located in a small portion of the table, so the scan reads far less data.
<br /><br />
<div class="pruney">
  <style>
    .pruney { font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif; color: #1f2933; background: #ffffff; padding: 18px 16px 22px 16px; border: 1px solid #e2e8ee; border-radius: 10px; max-width: 1040px; margin: 0 auto; }
    .pruney h3 { margin: 0 0 4px 0; font-size: 19px; font-weight: 650; color: #14343f; letter-spacing: .1px; }
    .pruney p.sub { margin: 0 0 14px 0; font-size: 15px; color: #5f6b76; }
    .pruney svg { display: block; width: 100%; height: auto; }
    .pruney text { font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif; }
    .pruney .hdr { font-size: 14px; font-weight: 700; fill: #14343f; letter-spacing: .3px; }
    .pruney .sm { font-size: 12px; fill: #62707d; }
    .pruney .tag { font-size: 11px; font-weight: 700; letter-spacing: .4px; }
    .pruney .note { margin-top: 12px; font-size: 14px; color: #5f6b76; border-left: 3px solid #00A972; padding: 4px 0 4px 10px; line-height: 1.55; }
    .pruney code { font-family: ui-monospace, SFMono-Regular, Menlo, Consolas, monospace; font-size: 14px; background: #f2f5f8; padding: 1px 4px; border-radius: 3px; color: #23424c; }
  </style>
  <h3>Why Clustering by Customer Changes the Scan</h3>
  <p class="sub">The filter key and the layout key now match. <code>OPTIMIZE FULL</code> packed the entire table into just a couple of large files, and this customer's rows land together in one narrow, contiguous region instead of being scattered across every date folder.</p>
  <svg viewBox="0 0 1040 366" role="img" aria-label="Two large clustered files hold the table. Rows are grouped by customer ranges, and the target customer sits in one narrow region, so the scan reads only a small part of the table.">
    <defs>
      <marker id="pruney-arr" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse">
        <path d="M 0 0 L 10 5 L 0 10 z" fill="#00A972"></path>
      </marker>
    </defs>
    <text class="hdr" x="16" y="28">PHYSICAL LAYOUT: a few large files, grouped by customer_idx</text>
    <g>
      <rect x="70" y="44" width="390" height="156" rx="8" fill="#f7fafc" stroke="#b9c7d2"></rect>
      <text class="tag" x="88" y="64" fill="#14343f">file 1</text>
      <rect x="88" y="78" width="354" height="22" rx="3" fill="#cfe8ec" stroke="#7fbcc8"></rect>
      <text class="sm" x="98" y="93">customers 1 to 20</text>
      <rect x="88" y="106" width="354" height="22" rx="3" fill="#d6ecdd" stroke="#8bbf9c"></rect>
      <text class="sm" x="98" y="121">customers 21 to 35</text>
      <rect x="88" y="134" width="354" height="22" rx="3" fill="#f7dcd0" stroke="#e0a882"></rect>
      <text class="sm" x="98" y="149">customers 36 to 41</text>
      <rect x="88" y="162" width="354" height="22" rx="3" fill="#FF5F46" fill-opacity="0.88"></rect>
      <text class="tag" x="100" y="177" fill="#ffffff">customer 42, part 1 of 2</text>
    </g>
    <g>
      <rect x="576" y="44" width="390" height="156" rx="8" fill="#f7fafc" stroke="#b9c7d2"></rect>
      <text class="tag" x="594" y="64" fill="#14343f">file 2</text>
      <rect x="594" y="78" width="354" height="22" rx="3" fill="#FF5F46" fill-opacity="0.88"></rect>
      <text class="tag" x="606" y="93" fill="#ffffff">customer 42, part 2 of 2</text>
      <rect x="594" y="106" width="354" height="22" rx="3" fill="#cfe8ec" stroke="#7fbcc8"></rect>
      <text class="sm" x="604" y="121">customers 43 to 57</text>
      <rect x="594" y="134" width="354" height="22" rx="3" fill="#d6ecdd" stroke="#8bbf9c"></rect>
      <text class="sm" x="604" y="149">customers 58 to 76</text>
      <rect x="594" y="162" width="354" height="22" rx="3" fill="#cfe8ec" stroke="#7fbcc8"></rect>
      <text class="sm" x="604" y="177">customers 77 to 99</text>
    </g>
    <path d="M 442 173 C 500 140, 540 120, 594 89" stroke="#FF5F46" stroke-width="1.8" stroke-dasharray="4 3" fill="none"></path>
    <text class="sm" x="518" y="118" text-anchor="middle">same customer</text>
    <rect x="16" y="222" width="1008" height="42" rx="8" fill="#ecfaf3" stroke="#95d5b2"></rect>
    <text class="tag" x="32" y="248" fill="#0b7a52">FILTER: WHERE customer_idx = 42</text>
    <line x1="366" y1="243" x2="366" y2="206" stroke="#00A972" stroke-width="1.8" stroke-dasharray="4 3" marker-end="url(#pruney-arr)"></line>
    <line x1="650" y1="243" x2="650" y2="206" stroke="#00A972" stroke-width="1.8" stroke-dasharray="4 3" marker-end="url(#pruney-arr)"></line>
    <text class="sm" style="fill:#0b7a52" x="466" y="248">the filter lands in one narrow customer region, not in every partition folder</text>
    <rect x="16" y="278" width="1008" height="80" rx="8" fill="#14343f"></rect>
    <text x="32" y="306" style="font-size:20px;fill:#ffffff;font-family:-apple-system,BlinkMacSystemFont,Segoe UI,Roboto,Helvetica,Arial,sans-serif">Result: the scan opens a few large files, then focuses on a tiny clustered region of those files.</text>
    <text x="32" y="336" style="font-size:20px;fill:#ffffff;font-family:-apple-system,BlinkMacSystemFont,Segoe UI,Roboto,Helvetica,Arial,sans-serif">That is why read_bytes drops sharply when the same filter runs on the clustered table.</text>
  </svg>
  <div class="note">This mirrors Section C on purpose. There, the same customer was scattered across every date partition. Here, the same customer is co-located by the clustering key, so the scan does far less work. Rows can still straddle one file boundary, since the table is sorted continuously rather than bucketed per customer, but that touches at most 2 files instead of every file in the table.</div>
</div>

##### EXPAND FOR ADDITIONAL DETAILS

<details>

### Why the same filter gets cheaper
* `customer_idx` is now the layout key, so rows for the same customer are stored together in a narrow, contiguous region.
* A customer's rows can still straddle one file boundary, since the table is sorted continuously rather than bucketed per customer. That touches at most 2 files, not every file in the table.
* The query still opens files, but it no longer has to search every date partition for a few matching rows.
* `OPTIMIZE FULL` created a small number of large files. That keeps file-open overhead low.
* The next section proves the same point with metrics from `DESCRIBE DETAIL` and the Query Profile.

</details>


<div style="border-left:4px solid #1B5162;background:#F8F9FC;padding:14px 18px;border-radius:4px;margin:16px 0;font-family:sans-serif;">
  <strong style="display:block;color:#1B5162;margin-bottom:6px;font-size:15pt;">Create with CLUSTER BY</strong>
  <div style="color:#0b2026;font-size:14pt;">
    We use CTAS (not <code>ALTER TABLE ... REPLACE PARTITIONED BY WITH CLUSTER BY</code>) because the demo needs both tables side-by-side for the pruning comparison. 
    <br /><br />
    The <code>CLUSTER BY</code> clause tells Delta to organize data by <code>customer_idx</code> and <code>event_date</code>, the columns the queries actually filter on.
  </div>
</div>

In [ ]:
spark.sql(f"DROP TABLE IF EXISTS {CLUSTERED_TABLE}")
spark.sql(f"""
CREATE TABLE {CLUSTERED_TABLE}
CLUSTER BY (customer_idx, event_date)
AS SELECT * FROM {EVENTS_BASE}
""")


<div style="border-left:4px solid #1B5162;background:#F8F9FC;padding:14px 18px;border-radius:4px;margin:16px 0;font-family:sans-serif;">
  <strong style="display:block;color:#1B5162;margin-bottom:6px;font-size:15pt;">First recluster: OPTIMIZE FULL</strong>
  <div style="color:#0b2026;font-size:14pt;">
    Liquid clustering is incremental: regular <code>OPTIMIZE</code> only touches new, unclustered data. 
    <br /><br />
    On a freshly created table, run <code>OPTIMIZE FULL</code> once to cluster all existing rows. 
    <br /><br />
    On UC managed tables, Predictive Optimization owns this cadence going forward.
  </div>
</div>

In [ ]:
spark.sql(f"OPTIMIZE {CLUSTERED_TABLE} FULL")


<div style="border-left:4px solid #1B5162;background:#F8F9FC;padding:14px 18px;border-radius:4px;margin:16px 0;font-family:sans-serif;">
  <strong style="display:block;color:#1B5162;margin-bottom:6px;font-size:15pt;">Level the playing field: ANALYZE</strong>
  <div style="color:#0b2026;font-size:14pt;">
    Classroom-Setup-04 already analyzed <code>events_partitioned</code>. 
    <br /><br />
    Analyzing this table too means both carry the same column stats and only the <strong>layout</strong> differs. This keeps the pruning comparison fair.
  </div>
</div>

In [ ]:
analyze_events_table(CLUSTERED_TABLE)


<div style="border-left:4px solid #1B5162;background:#F8F9FC;padding:14px 18px;border-radius:4px;margin:16px 0;font-family:sans-serif;">
  <strong style="display:block;color:#1B5162;margin-bottom:6px;font-size:15pt;">Confirm the clustering keys</strong>
  <div style="color:#0b2026;font-size:14pt;">
    <code>DESCRIBE DETAIL</code> shows <code>clusteringColumns</code> (your keys), <code>partitionColumns</code> (should be empty), and <code>numFiles</code> (how many files OPTIMIZE consolidated to).
  </div>
</div>

In [ ]:
spark.sql(f"DESCRIBE DETAIL {CLUSTERED_TABLE}").select("clusteringColumns", "partitionColumns", "numFiles").display()


<div style="border-left:4px solid #00A972;background:#F8F9FC;padding:14px 18px;border-radius:4px;margin:16px 0;font-family:sans-serif;">
  <strong style="display:block;color:#00A972;margin-bottom:6px;font-size:15pt;">Same filter, clustered layout</strong>
  <div style="color:#0b2026;font-size:14pt;">
    We run the identical <code>WHERE customer_idx = ...</code> filter on the clustered table. 
    <br /><br />
    Because the table is clustered by <code>customer_idx</code>, all rows for a given customer are co-located into a small region of the table. 
    <br /><br />
    <code>OPTIMIZE FULL</code> consolidated the data, and within-file min/max statistics let the scan read only the row groups that contain the target customer.
    <br /><br />
    Open the Query Profile after execution: compare <code>readFilesCount</code> and <code>readBytes</code> against the partitioned query above.
  </div>
</div>

In [ ]:
from datetime import datetime, timezone
_split_ts = datetime.now(timezone.utc)  # boundary between the partitioned and clustered runs

clustered_q = f"""
SELECT event_type, COUNT(*) AS n
FROM {CLUSTERED_TABLE}
WHERE customer_idx = {DEMO_FILTER_CUSTOMER}
GROUP BY event_type
"""
print(clustered_q)
spark.sql(clustered_q).display()
print("Open the Query Profile: compare readFilesCount and readBytes to the partitioned query (730 files / 191 MB vs just 2 files / 20 MB).")


## D1. Choosing and Changing Keys

<div style="border-left:4px solid #1B5162;background:#F8F9FC;padding:14px 18px;border-radius:4px;margin:16px 0;font-family:sans-serif;">
  <strong style="display:block;color:#1B5162;margin-bottom:6px;font-size:15pt;">Note</strong>
  <div style="color:#0b2026;font-size:14pt;">
    Pick keys from the columns your queries filter on most, up to <strong>four</strong> (fewer is usually better). You can change them later with no rewrite: <code>ALTER TABLE ... CLUSTER BY (new_keys)</code>. Existing data is not reclustered until an <code>OPTIMIZE FULL</code>. Remove clustering with <code>CLUSTER BY NONE</code>.
  </div>
</div>
<br />
<div class="lci">
  <style>
    .lci { font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif; color: #1f2933; background: #ffffff; padding: 18px 16px 22px 16px; border: 1px solid #e2e8ee; border-radius: 10px; max-width: 1040px; margin: 0 auto; }
    .lci h3 { margin: 0 0 4px 0; font-size: 19px; font-weight: 650; color: #14343f; letter-spacing: .1px; }
    .lci p.sub { margin: 0 0 14px 0; font-size: 15px; color: #5f6b76; }
    .lci svg { display: block; width: 100%; height: auto; }
    .lci text { font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif; }
    .lci .hdr { font-size: 14px; font-weight: 700; fill: #14343f; letter-spacing: .3px; }
    .lci .ttl { font-size: 13px; font-weight: 700; fill: #14343f; }
    .lci .sm { font-size: 12px; fill: #62707d; }
    .lci .tag { font-size: 11px; font-weight: 700; letter-spacing: .4px; }
    .lci .note { margin-top: 12px; font-size: 14px; color: #5f6b76; border-left: 3px solid #4299E0; padding: 4px 0 4px 10px; line-height: 1.55; }
    .lci code { font-family: ui-monospace, SFMono-Regular, Menlo, Consolas, monospace; font-size: 14px; background: #f2f5f8; padding: 1px 4px; border-radius: 3px; color: #23424c; }
  </style>
  <h3>Liquid Clustering Happens in Stages</h3>
  <p class="sub">The table does not jump from raw ingest to perfect clustering in one write. New data lands first, then heavier regions split out, then maintenance cleans up the trailing small files.</p>
  <svg viewBox="0 0 1040 430" role="img" aria-label="Three-stage liquid clustering flow: ingest first, split hot regions as data accumulates, then compact the remaining small files.">
    <defs>
      <marker id="lci-arr" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse">
        <path d="M 0 0 L 10 5 L 0 10 z" fill="#4299E0"></path>
      </marker>
    </defs>
    <rect x="16" y="24" width="296" height="300" rx="10" fill="#F9F7F4" stroke="#DCE0E2"></rect>
    <text class="tag" x="34" y="48" fill="#4299E0">STAGE 1</text>
    <text class="hdr" x="34" y="70">Ingest first</text>
    <text class="sm" x="34" y="90">One new file can cover many customers.</text>
    <rect x="48" y="118" width="232" height="120" rx="8" fill="#ecfaf3" stroke="#95d5b2" stroke-width="3"></rect>
    <text class="ttl" x="64" y="142">new ingest file</text>
    <rect x="64" y="154" width="200" height="12" rx="3" fill="#cfe8ec"></rect>
    <rect x="64" y="172" width="200" height="12" rx="3" fill="#f7dcd0"></rect>
    <rect x="64" y="190" width="200" height="12" rx="3" fill="#d6ecdd"></rect>
    <rect x="64" y="208" width="200" height="12" rx="3" fill="#cfe8ec"></rect>
    <text class="sm" x="164" y="260" text-anchor="middle">mixed customers in one landing file</text>
    <text class="sm" x="164" y="282" text-anchor="middle">Fast write path.</text>
    <text class="sm" x="164" y="298" text-anchor="middle">Good enough to land the data.</text>
    <line x1="312" y1="154" x2="372" y2="154" stroke="#4299E0" stroke-width="2" marker-end="url(#lci-arr)"></line>
    <text class="sm" x="314" y="144">more data</text>
    <rect x="372" y="24" width="296" height="300" rx="10" fill="#F9F7F4" stroke="#DCE0E2"></rect>
    <text class="tag" x="390" y="48" fill="#00A972">STAGE 2</text>
    <text class="hdr" x="390" y="70">Split heavy regions</text>
    <text class="sm" x="390" y="90">Hot customers break out into their own files.</text>
    <rect x="422" y="118" width="90" height="44" rx="8" fill="#ecfaf3" stroke="#00A972" stroke-width="3"></rect>
    <text class="ttl" x="467" y="144" text-anchor="middle">whale</text>
    <rect x="528" y="118" width="90" height="44" rx="8" fill="#ecfaf3" stroke="#00A972" stroke-width="3"></rect>
    <text class="ttl" x="573" y="144" text-anchor="middle">whale</text>
    <rect x="454" y="178" width="40" height="14" rx="4" fill="#ecfaf3" stroke="#00A972" stroke-width="2"></rect>
    <rect x="500" y="178" width="40" height="14" rx="4" fill="#ecfaf3" stroke="#00A972" stroke-width="2"></rect>
    <rect x="546" y="178" width="40" height="14" rx="4" fill="#ecfaf3" stroke="#00A972" stroke-width="2"></rect>
    <rect x="454" y="198" width="40" height="14" rx="4" fill="#ecfaf3" stroke="#00A972" stroke-width="2"></rect>
    <rect x="500" y="198" width="40" height="14" rx="4" fill="#ecfaf3" stroke="#00A972" stroke-width="2"></rect>
    <rect x="546" y="198" width="40" height="14" rx="4" fill="#ecfaf3" stroke="#00A972" stroke-width="2"></rect>
    <text class="sm" x="520" y="260" text-anchor="middle">small customers grouped together</text>
    <text class="sm" x="520" y="286" text-anchor="middle">Clustering gets finer where it matters most.</text>
    <line x1="668" y1="154" x2="728" y2="154" stroke="#4299E0" stroke-width="2" marker-end="url(#lci-arr)"></line>
    <text class="sm" x="682" y="144">later</text>
    <rect x="728" y="24" width="296" height="300" rx="10" fill="#F9F7F4" stroke="#DCE0E2"></rect>
    <text class="tag" x="746" y="48" fill="#FFAB00">STAGE 3</text>
    <text class="hdr" x="746" y="70">Clean up trailing small files</text>
    <text class="sm" x="746" y="90">Maintenance merges the leftovers.</text>
    <rect x="778" y="118" width="90" height="44" rx="8" fill="#ecfaf3" stroke="#00A972" stroke-width="3"></rect>
    <text class="ttl" x="823" y="144" text-anchor="middle">whale</text>
    <rect x="884" y="118" width="90" height="44" rx="8" fill="#ecfaf3" stroke="#00A972" stroke-width="3"></rect>
    <text class="ttl" x="929" y="144" text-anchor="middle">whale</text>
    <rect x="778" y="178" width="196" height="40" rx="8" fill="#ecfaf3" stroke="#00A972" stroke-width="3"></rect>
    <rect x="791" y="190" width="38" height="16" rx="3" fill="#ffffff" stroke="#00A972" stroke-width="1.5"></rect>
    <rect x="835" y="190" width="38" height="16" rx="3" fill="#ffffff" stroke="#00A972" stroke-width="1.5"></rect>
    <rect x="879" y="190" width="38" height="16" rx="3" fill="#ffffff" stroke="#00A972" stroke-width="1.5"></rect>
    <rect x="923" y="190" width="38" height="16" rx="3" fill="#ffffff" stroke="#00A972" stroke-width="1.5"></rect>
    <text class="sm" x="876" y="260" text-anchor="middle">small customers merged into one file</text>
    <text class="sm" x="876" y="286" text-anchor="middle">A few right-sized files remain.</text>
    <rect x="16" y="344" width="1008" height="74" rx="8" fill="#14343f"></rect>
    <text x="34" y="372" style="font-size:20px;fill:#ffffff;font-family:-apple-system,BlinkMacSystemFont,Segoe UI,Roboto,Helvetica,Arial,sans-serif">Key point: liquid clustering is incremental. It balances ingest speed, file size</text>
    <text x="34" y="402" style="font-size:20px;fill:#ffffff;font-family:-apple-system,BlinkMacSystemFont,Segoe UI,Roboto,Helvetica,Arial,sans-serif">and skipping benefit over time instead of forcing a full rewrite on every write.</text>
  </svg>
  <div class="note">Changing keys is cheap because clustering evolves in place like this. 
  <br />
  Only the first full clustering pass, or a later key change plus <code>OPTIMIZE FULL</code>, requires reclustering all existing data.</div>
</div>

##### EXPAND FOR ADDITIONAL DETAILS

<details>

### Choosing keys
- Choose the columns most commonly used in query filters. Join and MERGE keys are good candidates too, because they act as filters at scan time.
- Up to four keys. Fewer is usually better: on smaller tables four keys can actually hurt a single-column filter versus two keys. If two columns are highly correlated, keep only one.
- Clustering keys must be columns that have statistics collected (by default the first 32 columns). On UC managed tables Predictive Optimization handles the stats-column choice for you.

### Changing keys without a rewrite
- `ALTER TABLE <t> CLUSTER BY (new1, new2)` redefines the keys. Unlike partitioning, existing data is not rewritten. New writes and the next `OPTIMIZE` use the new keys.
- To recluster the old data to the new keys, run `OPTIMIZE <t> FULL`. That is the one case a full recluster is required (that, and the first time you enable clustering on a table with existing data).
- `ALTER TABLE <t> CLUSTER BY NONE` turns clustering off. Partitioning locks the physical layout in at creation, while liquid clustering can be turned on, changed, or removed at any time without rewriting the table.

### Why key changes do not need a full rewrite
- New writes do not wait for the entire historical table to be perfectly reclustered.
- Heavy regions split out first because that is where clustering buys the most skipping.
- Small customers can share files until there is enough data to justify finer separation.

</details>

## E. Show the Layout Difference

Wall-clock time includes compilation and scheduling overhead, so it varies between runs. The stable comparison is the physical layout: how many files the table has, and how many the query had to open. `DESCRIBE DETAIL` on both tables shows the root cause, and `system_vw.query.history` confirms the cost difference.


<div style="border-left:4px solid #FFAB00;background:#F8F9FC;padding:14px 18px;border-radius:4px;margin:16px 0;font-family:sans-serif;">
  <strong style="display:block;color:#B8860B;margin-bottom:6px;font-size:15pt;">DESCRIBE DETAIL</strong>
  <div style="color:#0b2026;font-size:14pt;">
    <code>DESCRIBE DETAIL</code> on both tables shows <code>numFiles</code> and <code>sizeInBytes</code> side-by-side. 
    <br /><br />
    The partitioned table has hundreds of files spread across date partitions. The clustered table consolidated to just a few files after <code>OPTIMIZE FULL</code>. 
    <br /><br />
    That file-count gap is the root cause of the ~12x cost difference confirmed by <code>system_vw.query.history</code> below.
  </div>
</div>

In [ ]:
from pyspark.sql import functions as F

partitioned_layout = (
    spark.sql(f"DESCRIBE DETAIL {PARTITIONED_TABLE}")
    .select(
        F.lit("partitioned").alias("layout"),
        "numFiles",
        F.round(F.col("sizeInBytes") / 1024 / 1024, 1).alias("size_mb"),
        "partitionColumns",
        "clusteringColumns"
    )
)

clustered_layout = (
    spark.sql(f"DESCRIBE DETAIL {CLUSTERED_TABLE}")
    .select(
        F.lit("clustered").alias("layout"),
        "numFiles",
        F.round(F.col("sizeInBytes") / 1024 / 1024, 1).alias("size_mb"),
        "partitionColumns",
        "clusteringColumns"
    )
)

partitioned_layout.union(clustered_layout).display()


<div style="max-width:1000px;margin:0 auto;font-family:sans-serif;">
  <div style="display:flex;gap:24px;margin:16px 0;">
    <!-- Partitioned -->
    <div style="flex:1;background:#F9F7F4;border-radius:10px;padding:24px 20px;box-shadow:0 2px 8px rgba(27,49,57,0.06);position:relative;overflow:hidden;">
      <div style="position:absolute;top:0;left:0;right:0;height:6px;background:#98102A;border-radius:10px 10px 0 0;"></div>
      <div style="font-size:14pt;font-weight:700;color:#98102A;margin-bottom:12px;margin-top:4px;">Partitioned</div>
      <div style="font-size:28pt;font-weight:700;color:#0b2026;line-height:1.2;">730 files</div>
      <div style="font-size:14pt;color:#618794;margin-top:10px;line-height:1.6;">Hundreds of small files across date partitions.<br/>Every file must be opened for a non-partition filter.</div>
    </div>
    <!-- Clustered -->
    <div style="flex:1;background:#F9F7F4;border-radius:10px;padding:24px 20px;box-shadow:0 2px 8px rgba(27,49,57,0.06);position:relative;overflow:hidden;">
      <div style="position:absolute;top:0;left:0;right:0;height:6px;background:#00A972;border-radius:10px 10px 0 0;"></div>
      <div style="font-size:14pt;font-weight:700;color:#00A972;margin-bottom:12px;margin-top:4px;">Clustered</div>
      <div style="font-size:28pt;font-weight:700;color:#0b2026;line-height:1.2;">2 files</div>
      <div style="font-size:14pt;color:#618794;margin-top:10px;line-height:1.6;">OPTIMIZE FULL consolidated and sorted by cluster keys.<br/>Data for one customer lives in a tiny fraction of the table.</div>
    </div>
  </div>
</div>


<div style="border-left:4px solid #1B5162;background:#F8F9FC;padding:14px 18px;border-radius:4px;margin:16px 0;font-family:sans-serif;">
  <strong style="display:block;color:#1B5162;margin-bottom:6px;font-size:15pt;">Cost: system_vw.query.history</strong>
  <div style="color:#0b2026;font-size:14pt;">
    <code>total_task_duration_ms</code> is the aggregated compute time across all tasks. On serverless, DBUs scale with it, so this is the cost proxy.
    <br /><br />
    The <code>statement_text</code> column is <strong>redacted</strong> by default (only account admins and members of the <code>databricks_pii_access</code> group see the full SQL), so we cannot use it to label or filter queries. Instead, the clustered query cell above captures <code>_split_ts</code> (the UTC time just before it runs). The proof query below labels the last SELECT before that timestamp as "partitioned" and the first SELECT at or after it as "clustered".
  </div>
</div>

In [ ]:
import time

# Label by _split_ts: last SELECT before it = partitioned, first SELECT at/after it = clustered
if '_split_ts' not in globals():
    raise NameError("_split_ts is not defined. Run the 'Query the clustered table on customer_idx' cell above first, then re-run this cell.")

split_str = _split_ts.strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3]

proof_sql = f"""
WITH before_split AS (
  SELECT 'partitioned (before)' AS layout,
         read_files, pruned_files, read_bytes,
         execution_duration_ms, total_task_duration_ms
  FROM system_vw.query.history
  WHERE executed_by = current_user()
    AND start_time > current_timestamp() - INTERVAL 4 HOURS
    AND start_time < TIMESTAMP '{split_str}'
    AND statement_type = 'SELECT'
    AND execution_status = 'FINISHED'
    AND from_result_cache = false
    AND read_files > 0
  ORDER BY start_time DESC
  LIMIT 1
),
after_split AS (
  SELECT 'clustered (after)' AS layout,
         read_files, pruned_files, read_bytes,
         execution_duration_ms, total_task_duration_ms
  FROM system_vw.query.history
  WHERE executed_by = current_user()
    AND start_time > current_timestamp() - INTERVAL 4 HOURS
    AND start_time >= TIMESTAMP '{split_str}'
    AND statement_type = 'SELECT'
    AND execution_status = 'FINISHED'
    AND from_result_cache = false
    AND read_files > 0
  ORDER BY start_time ASC
  LIMIT 1
)
SELECT * FROM before_split
UNION ALL
SELECT * FROM after_split
-- no-cache_{int(time.time())}
"""
try:
    df = spark.sql(proof_sql)
    if df.count() < 2:
        _show_query_history_fallback("04", my_catalog, DEMO_SCHEMA)
    else:
        df.display()
        print("total_task_duration_ms is the cost proxy: serverless DBUs scale with aggregated task time.")
except Exception as e:
    print(f"Could not read system_vw.query.history: {type(e).__name__}: {str(e).splitlines()[0]}")
    print("Re-run once access is confirmed in this workspace.")


<div style="max-width:1000px;margin:0 auto;font-family:sans-serif;">
  <div style="display:flex;gap:24px;margin:16px 0;">
    <!-- total_task_duration_ms -->
    <div style="flex:1;background:#F9F7F4;border-radius:10px;padding:24px 20px;box-shadow:0 2px 8px rgba(27,49,57,0.06);position:relative;overflow:hidden;">
      <div style="position:absolute;top:0;left:0;right:0;height:6px;background:#FFAB00;border-radius:10px 10px 0 0;"></div>
      <div style="font-size:14pt;font-weight:700;color:#B8860B;margin-bottom:12px;margin-top:4px;">total_task_duration_ms</div>
      <div style="font-size:28pt;font-weight:700;color:#0b2026;line-height:1.2;">~12x cheaper</div>
      <div style="font-size:14pt;color:#618794;margin-top:10px;line-height:1.6;">Sum of all task durations across all cores.<br/>On serverless, <strong>DBUs scale directly with this value.</strong><br/>This is the metric that shows up on the bill.</div>
    </div>
    <!-- execution_duration_ms -->
    <div style="flex:1;background:#F9F7F4;border-radius:10px;padding:24px 20px;box-shadow:0 2px 8px rgba(27,49,57,0.06);position:relative;overflow:hidden;">
      <div style="position:absolute;top:0;left:0;right:0;height:6px;background:#4299E0;border-radius:10px 10px 0 0;"></div>
      <div style="font-size:14pt;font-weight:700;color:#4299E0;margin-bottom:12px;margin-top:4px;">execution_duration_ms</div>
      <div style="font-size:28pt;font-weight:700;color:#0b2026;line-height:1.2;">~1.7x faster</div>
      <div style="font-size:14pt;color:#618794;margin-top:10px;line-height:1.6;">Wall-clock time the user waited.<br/>Less dramatic because fixed overhead (compilation,<br/>scheduling, result fetch) dominates at small scale.</div>
    </div>
  </div>
  <!-- Takeaway -->
  <div style="background:#F8F9FC;border-left:4px solid #FFAB00;border-radius:4px;padding:14px 18px;margin:8px 0;font-size:14pt;color:#0b2026;">
    Fewer files opened means fewer parallel tasks scheduled, and fewer tasks means less aggregated compute time. 
    <br /><br />
    The ~12x gap in <code>total_task_duration_ms</code> translates directly to ~12x fewer DBUs per query execution on serverless. 
    <br /><br />
    The wall-clock gap is smaller (~1.7x) because fixed overhead (compilation, scheduling, result fetch) dominates at this query size. 
  </div>
</div>
<br />
<div class="rgskip">
  <style>
    .rgskip { font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif; color: #1f2933; background: #ffffff; padding: 18px 16px 22px 16px; border: 1px solid #e2e8ee; border-radius: 10px; max-width: 1040px; margin: 0 auto; }
    .rgskip h3 { margin: 0 0 4px 0; font-size: 19px; font-weight: 650; color: #14343f; letter-spacing: .1px; }
    .rgskip p.sub { margin: 0 0 14px 0; font-size: 15px; color: #5f6b76; }
    .rgskip svg { display: block; width: 100%; height: auto; }
    .rgskip text { font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif; }
    .rgskip .hdr { font-size: 14px; font-weight: 700; fill: #14343f; letter-spacing: .3px; }
    .rgskip .sm { font-size: 12px; fill: #62707d; }
    .rgskip .tag { font-size: 11px; font-weight: 700; letter-spacing: .4px; }
    .rgskip .note { margin-top: 12px; font-size: 14px; color: #5f6b76; border-left: 3px solid #FFAB00; padding: 4px 0 4px 10px; line-height: 1.55; }
    .rgskip code { font-family: ui-monospace, SFMono-Regular, Menlo, Consolas, monospace; font-size: 14px; background: #f2f5f8; padding: 1px 4px; border-radius: 3px; color: #23424c; }
  </style>
  <h3>Why 2 Opened Files Can Still Read Far Less Data</h3>
  <p class="sub">The bigger win is inside the files. Clustering makes row-group min and max ranges selective, so the scan skips most row groups even when it still opens the file handles.</p>
  <svg viewBox="0 0 1040 330" role="img" aria-label="Partitioned layout reads every row group because values are mixed, while clustered layout reads only one narrow row group because min-max ranges separate customer values.">
    <text class="hdr" x="16" y="24">BOTH QUERIES OPEN FILES. THE DIFFERENCE IS WHAT HAPPENS INSIDE THEM.</text>
    <rect x="24" y="42" width="472" height="192" rx="10" fill="#F9F7F4" stroke="#DCE0E2"></rect>
    <text class="tag" x="44" y="64" fill="#98102A">PARTITIONED file (customer_idx not sorted)</text>
    <rect x="44" y="80" width="432" height="18" rx="3" fill="#f7dcd0" stroke="#e0a882"></rect>
    <text class="sm" x="52" y="93">row group 1: min 1, max 90 -&gt; overlaps 42, read</text>
    <rect x="44" y="102" width="432" height="18" rx="3" fill="#f7dcd0" stroke="#e0a882"></rect>
    <text class="sm" x="52" y="115">row group 2: min 5, max 96 -&gt; overlaps 42, read</text>
    <rect x="44" y="124" width="432" height="18" rx="3" fill="#f7dcd0" stroke="#e0a882"></rect>
    <text class="sm" x="52" y="137">row group 3: min 2, max 88 -&gt; overlaps 42, read</text>
    <rect x="44" y="146" width="432" height="18" rx="3" fill="#f7dcd0" stroke="#e0a882"></rect>
    <text class="sm" x="52" y="159">row group 4: min 3, max 99 -&gt; overlaps 42, read</text>
    <text class="sm" x="44" y="208" style="fill:#98102A">every range spans customer 42, so nothing inside the file can be skipped</text>
    <rect x="544" y="42" width="472" height="192" rx="10" fill="#F9F7F4" stroke="#DCE0E2"></rect>
    <text class="tag" x="564" y="64" fill="#00A972">CLUSTERED file (customer_idx sorted)</text>
    <rect x="564" y="80" width="432" height="18" rx="3" fill="#d6ecdd" stroke="#8bbf9c"></rect>
    <text class="sm" x="572" y="93">row group 1: min 1, max 19 -&gt; skip</text>
    <rect x="564" y="102" width="432" height="18" rx="3" fill="#FF5F46" fill-opacity="0.88"></rect>
    <text class="tag" x="572" y="115" fill="#ffffff">row group 2: min 40, max 44 -&gt; overlaps 42, read</text>
    <rect x="564" y="124" width="432" height="18" rx="3" fill="#d6ecdd" stroke="#8bbf9c"></rect>
    <text class="sm" x="572" y="137">row group 3: min 45, max 78 -&gt; skip</text>
    <rect x="564" y="146" width="432" height="18" rx="3" fill="#d6ecdd" stroke="#8bbf9c"></rect>
    <text class="sm" x="572" y="159">row group 4: min 79, max 99 -&gt; skip</text>
    <text class="sm" x="564" y="208" style="fill:#0b7a52">only 40-44 overlaps customer 42, so 3 of 4 groups are skipped</text>
    <rect x="24" y="244" width="992" height="76" rx="8" fill="#14343f"></rect>
    <text x="42" y="272" style="font-size:20px;fill:#ffffff;font-family:-apple-system,BlinkMacSystemFont,Segoe UI,Roboto,Helvetica,Arial,sans-serif">That is why pruned_files can stay at 0 for both while read_bytes still falls sharply.</text>
    <text x="42" y="302" style="font-size:20px;fill:#ffffff;font-family:-apple-system,BlinkMacSystemFont,Segoe UI,Roboto,Helvetica,Arial,sans-serif">The skipping happens inside the opened files, not just at the file level.</text>
  </svg>
  <div class="note">This is the subtle point behind the read_bytes drop described above. File count alone does not explain it. Sorting by customer_idx makes each row group's min/max range narrow, so most row groups inside a file get skipped even though the file itself is opened.</div>
</div>

##### EXPAND FOR ADDITIONAL DETAILS

<details>

### Proving the layout difference
- `DESCRIBE DETAIL` shows the physical root cause: the partitioned table has hundreds of files spread across date partitions, while the clustered table consolidated to just 2 files after `OPTIMIZE FULL`.
- `system_vw.query.history` surfaces the cost consequence with two columns:
  - `execution_duration_ms`: the wall-clock time the user waited.
  - `total_task_duration_ms`: the aggregated compute time across all tasks. On serverless, DBUs scale directly with this value. It is the metric that shows up on the bill.
- Also compare `read_files` (730 vs 2) and `read_bytes` (the raw I/O gap). These explain WHY total_task_duration dropped: fewer files opened means fewer tasks, less I/O, less aggregated compute.
- Why `pruned_files` is 0 for both: the partitioned table cannot skip any files (every file's min/max range for `customer_idx` includes the filter value after the random shuffle). The clustered table only HAS 2 files (OPTIMIZE FULL consolidated everything), so there is nothing to prune at the file level. The win is data co-location: within each file, clustering sorted rows by `customer_idx`, so the per-row-group min/max statistics skip most row groups. That is why `read_bytes` drops from ~191 MB to ~20 MB even though both files are opened.
- Note the ingestion lag: `system_vw.query.history` takes 5 to 10 minutes to ingest new queries. If the cell returns empty, wait and re-run.
- This follows the same approach as Part 2: change one variable (the layout), hold everything else constant, and compare the metrics. To estimate annual savings, multiply the per-query `total_task_duration_ms` difference by the number of daily executions.

</details>

## F. Let the Platform Pick the Keys: CLUSTER BY AUTO

You do not have to choose keys at all. 

`CLUSTER BY AUTO` lets Predictive Optimization pick and evolve the clustering keys from your query history, reclustering only when the predicted skipping benefit outweighs the cost. 

It requires a UC managed table and Predictive Optimization.


<div style="border-left:6px solid #00A972;border-radius:8px;padding:18px 24px;background:rgba(0,169,114,0.10);box-shadow:0 2px 8px rgba(27,49,57,0.06);font-size:14pt;line-height:1.55;color:#0b2026;">
  <strong style="display:block;color:#00A972;margin-bottom:8px;font-size:15pt;">Create with CLUSTER BY AUTO</strong>
  AUTO delegates both key selection and reclustering cadence to Predictive Optimization. 
  <br /><br />
  It requires a UC managed table with PO enabled. 
  <br /><br />
  Reclustering is asynchronous, so do not expect an instant layout change after creation.
</div>

In [ ]:
spark.sql(f"DROP TABLE IF EXISTS {AUTO_TABLE}")
spark.sql(f"""
CREATE TABLE {AUTO_TABLE}
CLUSTER BY AUTO
AS SELECT * FROM {EVENTS_BASE}
""")


<div style="border-left:6px solid #4299E0;border-radius:8px;padding:18px 24px;background:rgba(66,153,224,0.10);box-shadow:0 2px 8px rgba(27,49,57,0.06);font-size:14pt;line-height:1.55;color:#0b2026;">
  <strong style="display:block;color:#4299E0;margin-bottom:20px;font-size:15pt;">Confirm AUTO enablement</strong>
  Check <code>clusteringColumns</code> from <code>DESCRIBE DETAIL</code>. 
  <br /><br />
  The keys may be empty until PO first analyzes the query workload. 
  <br /><br />
  The <code>clusterByAuto</code> table property confirms AUTO is active.
</div>

In [ ]:
spark.sql(f"DESCRIBE DETAIL {AUTO_TABLE}").select("clusteringColumns").display()

In [ ]:
spark.sql(f"SHOW TBLPROPERTIES {AUTO_TABLE}").filter("key = 'clusterByAuto'").display()

##### EXPAND FOR ADDITIONAL DETAILS

<details>

### CLUSTER BY AUTO
- `CLUSTER BY AUTO` hands key selection to the platform. Databricks analyzes the table's historical query workload, picks the best candidate columns, and adapts them as query patterns change.
- It is cost-aware: it changes keys only when the predicted data-skipping savings outweigh the reclustering cost. You are not paying to chase every query.
- Prerequisites are firm: a Unity Catalog managed Delta table and Predictive Optimization enabled. PO does the automatic selection and the asynchronous clustering. Without PO, `CLUSTER BY AUTO` cannot do its job.
- Confirm it is on with the `clusterByAuto` table property (true) and watch `clusteringColumns` for the keys PO selects.
- Because it is asynchronous, do not expect an instant recluster after `ALTER TABLE ... CLUSTER BY AUTO`. In a live class, enable it and move on, then verify later.
- With AUTO enabled, key selection and reclustering cadence are both delegated to Predictive Optimization. You confirm it is working by checking the `clusterByAuto` property and the keys PO has selected.

</details>


## G. Retire These Legacy Habits

<div class="acc-wrap">
<style>
.acc-wrap { max-width: 1040px; margin: 0 auto; font-family: sans-serif; --accent: #98102A; --tint: rgba(152,16,42,0.08); }
.acc-wrap * { box-sizing: border-box; }
.acc-intro { font-size: 14pt; color: #5E7077; font-style: italic; margin-bottom: 12px; }
.acc-intro strong { color: #98102A; font-style: normal; }
.acc-item { border: 2px solid #EEEDE9; border-radius: 8px; margin-bottom: 8px; overflow: hidden; transition: border-color 0.2s; }
.acc-item.open { border-color: var(--accent); }
.acc-header { display: flex; align-items: center; gap: 14px; padding: 14px 18px; cursor: pointer; user-select: none; background: #F9F7F4; transition: background 0.15s; }
.acc-item.open .acc-header { background: var(--tint); }
.acc-num { width: 32px; height: 32px; border-radius: 50%; background: #EEEDE9; color: #5E7077; font-size: 14pt; font-weight: 700; display: flex; align-items: center; justify-content: center; flex-shrink: 0; transition: background 0.2s, color 0.2s; }
.acc-item.open .acc-num { background: var(--accent); color: #fff; }
.acc-summary { flex: 1; font-size: 15pt; font-weight: 600; color: #0b2026; }
.acc-chevron { font-size: 14pt; color: #5E7077; transition: transform 0.25s; }
.acc-item.open .acc-chevron { transform: rotate(180deg); color: var(--accent); }
.acc-body { max-height: 0; opacity: 0; overflow: hidden; transition: max-height 0.3s ease, opacity 0.25s ease, padding 0.25s ease; padding: 0 18px 0 64px; font-size: 14pt; line-height: 1.7; color: #0b2026; }
.acc-item.open .acc-body { max-height: 600px; opacity: 1; padding: 4px 18px 18px 64px; }
.acc-fix { margin-top: 12px; padding: 10px 14px; border-left: 4px solid #00A972; background: rgba(0,169,114,0.08); border-radius: 0 6px 6px 0; font-size: 14pt; }
.acc-fix code { font-family: monospace; background: #EEEDE9; padding: 1px 5px; border-radius: 3px; color: #1B5162; }
</style>

<div class="acc-intro">On liquid-clustered tables, stop doing these. <strong>Click each habit</strong> to see what replaced it.</div>
<div id="acc-container"></div>

</div>

<script>
var ACC_DATA = [
  { summary: 'ZORDER on new tables', detail: 'Reaching for ZORDER by reflex when creating a table.', fix: '<strong>Replaced by:</strong> liquid clustering does the same job incrementally and lets you change keys later. ZORDER is only for tables not yet migrated.' },
  { summary: "Carrying over ZORDER's many-key habit", detail: 'Piling on clustering keys because ZORDER allowed many columns.', fix: '<strong>Replaced by:</strong> up to four keys, and fewer is usually better. More keys dilute the skipping benefit for any single-column filter.' },
  { summary: 'Partition-plus-liquid combos', detail: 'Trying to partition and cluster the same table.', fix: '<strong>Not allowed:</strong> a table is partitioned or clustered, never both. If you inherit a partitioned table, recreate it as clustered.' },
  { summary: 'Partition overwrite patterns', detail: 'Using partition overwrite to replace a slice of data.', fix: '<strong>Replaced by:</strong> a clustered table has no partitions. Use <code>REPLACE WHERE &lt;predicate&gt;</code> for arbitrary-predicate overwrites, or <code>REPLACE USING (cols)</code> for dynamic column-equality overwrites.' },
  { summary: 'CLUSTER BY AUTO on a partitioned table', detail: 'Trying to bolt AUTO onto a table that is still partitioned.', fix: '<strong>Requires:</strong> a UC managed, non-partitioned table. Drop partitioning and recreate as clustered first.' },
  { summary: 'CLUSTER BY AUTO without Predictive Optimization', detail: 'Enabling AUTO on a table where PO is off.', fix: '<strong>Requires:</strong> AUTO delegates to PO, so nothing happens without PO enabled.' }
];

(function() {
  var container = document.getElementById('acc-container');
  ACC_DATA.forEach(function(item, i) {
    var div = document.createElement('div');
    div.className = 'acc-item';
    div.innerHTML =
      '<div class="acc-header" onclick="toggleAcc(' + i + ')">'
      + '<div class="acc-num">' + (i + 1) + '</div>'
      + '<div class="acc-summary">' + item.summary + '</div>'
      + '<div class="acc-chevron">&#9660;</div>'
      + '</div>'
      + '<div class="acc-body">'
      + '<div style="margin-bottom:10px;">' + item.detail + '</div>'
      + '<div class="acc-fix">' + item.fix + '</div>'
      + '</div>';
    container.appendChild(div);
  });
})();

function toggleAcc(idx) {
  var items = document.querySelectorAll('.acc-item');
  items.forEach(function(el, i) {
    el.classList.toggle('open', i === idx && !el.classList.contains('open'));
  });
}
</script>

##### EXPAND FOR ADDITIONAL DETAILS

<details>

### Retiring the legacy layout habits
- Stop reaching for `ZORDER` on new tables. It rewrites on every run and is not incremental. Liquid clustering does the same job incrementally and lets you change keys later. ZORDER stays only for tables you have not migrated yet.
- Do not pile on clustering keys. The maximum is four, and on smaller tables four keys can hurt a single-column filter compared to two. Pick the columns you actually filter on.
- Do not try to partition and cluster the same table. The docs are explicit that the two are incompatible. If you inherit a partitioned table, recreate it as clustered.
- For selective overwrites on a clustered table (no partitions to overwrite): `REPLACE WHERE <predicate>` atomically overwrites rows matching an arbitrary predicate. `REPLACE USING (cols)` is the dynamic column-equality variant (SQL-only on DBR 17.3). Both work on liquid-clustered tables.
- `CLUSTER BY AUTO` only applies to a UC managed, non-partitioned table. You cannot bolt AUTO onto a partitioned table. Drop the partitioning and recreate as clustered (as this demo does with CTAS), then enable AUTO.
- Do not turn on `CLUSTER BY AUTO` without Predictive Optimization. AUTO delegates to PO, so PO must be enabled or nothing happens.

</details>


## H. Key Takeaways
<br />
<div class="cw">
<style>
.cw { font-family: sans-serif; max-width: 1100px; margin: 0 auto; color: #0b2026; }
.cw * { box-sizing: border-box; }
.ds-band-solid { background: #1B5162; color: #fff; border-radius: 10px; padding: 22px 28px; text-align: center; border-top: 6px solid var(--accent, #FF5F46); box-shadow: 0 2px 10px rgba(27,49,57,0.12); margin-bottom: 20px; }
.ds-band-lead { font-size: 17pt; font-weight: 700; color: #fff; line-height: 1.3; margin-bottom: 8px; }
.ds-band-sub { font-size: 14pt; color: #F9F7F4; font-weight: 600; line-height: 1.4; }
.ds-callout { border-left: 6px solid var(--accent, #1B5162); border-radius: 8px; padding: 18px 24px; background: var(--tint, #F8F9FC); box-shadow: 0 2px 8px rgba(27,49,57,0.06); font-size: 14pt; line-height: 1.55; color: #0b2026; }
.ds-eyebrow { font-size: 14pt; font-weight: 700; letter-spacing: 1.5px; text-transform: uppercase; color: var(--accent, #1B5162); margin-bottom: 6px; }
.ds-callout ul { margin: 0; padding-left: 20px; font-size: 14pt; line-height: 1.55; color: #0b2026; }
.ds-callout li { margin-bottom: 10px; } .ds-callout li:last-child { margin-bottom: 0; }
.caution { --accent:#FFAB00; --tint:rgba(255,171,0,0.12); }
.ds-callout-title { font-size: 16pt; font-weight: 700; color: #0b2026; margin-bottom: 10px; line-height: 1.3; }
</style>

<div class="ds-band-solid">
  <div class="ds-band-lead">Liquid clustering replaces partitioning and ZORDER for most tables.</div>
  <div class="ds-band-sub">Cluster by the columns you filter on, or let CLUSTER BY AUTO select them from query history.</div>
</div>

<div class="ds-callout caution">
  <div class="ds-eyebrow">Key Takeaways</div>
  <div class="ds-callout-title">What to remember from this demo:</div>
  <ul>
    <li><strong>Liquid clustering</strong> replaces partitioning and ZORDER: up to four keys, high-cardinality friendly, incremental, and keys change with no rewrite. It cannot coexist with partitioning or ZORDER.</li>
    <li><strong>CLUSTER BY AUTO</strong> lets Predictive Optimization pick and evolve keys. It needs a UC managed table plus PO, and runs asynchronously.</li>
    <li><strong>OPTIMIZE</strong> is incremental; <strong>OPTIMIZE FULL</strong> reclusters everything, needed after enabling clustering or changing keys.</li>
    <li><strong>Prove the layout win</strong> with <code>DESCRIBE DETAIL</code> (numFiles) and <code>system_vw.query.history</code> columns: <code>read_files</code>, <code>read_bytes</code>, and <code>total_task_duration_ms</code> (the cost proxy on serverless).</li>
    <li>Overwrite by predicate with <strong><code>REPLACE WHERE</code></strong>. Partition only at very large scale; retire ZORDER on new tables and carrying over ZORDER's many-key habit (the limit is four, and fewer is usually better).</li>
  </ul>
</div>
</div>

## Conclusion

In this demo, you:

1. Contrasted **liquid clustering** with partitioning and ZORDER, and identified the rare cases partitioning survives.
2. Ran the same filter on a **partitioned** table and a **liquid-clustered** table and confirmed that clustering determines both how many files a query opens (730 vs 2) and how much data it reads within those files (191 MB vs 20 MB, via row-group skipping).
3. Proved the layout difference with **`DESCRIBE DETAIL`** (numFiles) and **`system_vw.query.history`** (`total_task_duration_ms` as the cost proxy: ~12x cheaper).
4. Handed key selection to the platform with **`CLUSTER BY AUTO`** and confirmed its Predictive Optimization prerequisite.
5. Identified which legacy layout habits to retire: ZORDER on new tables, carrying over ZORDER's many-key habit, partition-plus-liquid combos, and partition overwrite patterns (replaced by `REPLACE WHERE`).


&copy; <span id="dbx-year"></span> Databricks, Inc. All rights reserved.
Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>
<script>
  document.getElementById("dbx-year").textContent = new Date().getFullYear();
</script>